# Treebank Re-parsing Comparison

Re-parse UD treebanks (Irish IDT, Swedish Talbanken, Swedish LinES) with Stanza and UDPipe 2,
compare outputs against gold annotations, and cross-compare parsers to identify automatically correctable errors.

## 1. Setup

In [ ]:
!pip install -q stanza conllu pandas matplotlib seaborn requests

In [ ]:
import stanza
import conllu
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import os
import re
import io
from collections import defaultdict, Counter
from pathlib import Path
from typing import Optional

sns.set_theme(style="whitegrid")

UDPIPE_API = "https://lindat.mff.cuni.cz/services/udpipe/api"

In [ ]:
# Download Stanza models
stanza.download("ga", package="idt")
stanza.download("sv", package="talbanken")

In [ ]:
# Check available UDPipe 2 models
resp = requests.get(f"{UDPIPE_API}/models").json()
udpipe_models = resp.get("models", {})
print("Available UDPipe models (Irish/Swedish):")
for m in sorted(udpipe_models.keys()) if isinstance(udpipe_models, dict) else sorted(udpipe_models):
    ml = m.lower()
    if "irish" in ml or "swedish" in ml or "gaelic" in ml:
        print(f"  {m}")

## 2. Load Gold Treebanks

In [ ]:
TREEBANKS = {
    "ga_idt": {
        "repo": "universaldependencies/UD_Irish-IDT",
        "dir": "UD_Irish-IDT",
        "test_file": "ga_idt-ud-test.conllu",
        "stanza_lang": "ga",
        "stanza_package": "idt",
        "udpipe_model": "irish-idt",
    },
    "sv_talbanken": {
        "repo": "universaldependencies/UD_Swedish-Talbanken",
        "dir": "UD_Swedish-Talbanken",
        "test_file": "sv_talbanken-ud-test.conllu",
        "stanza_lang": "sv",
        "stanza_package": "talbanken",
        "udpipe_model": "swedish-talbanken",
    },
    "sv_lines": {
        "repo": "universaldependencies/UD_Swedish-LinES",
        "dir": "UD_Swedish-LinES",
        "test_file": "sv_lines-ud-test.conllu",
        "stanza_lang": "sv",
        "stanza_package": "talbanken",  # LinES-specific model may not exist
        "udpipe_model": "swedish-lines",
    },
}

In [ ]:
# Clone treebank repos
for name, info in TREEBANKS.items():
    d = info["dir"]
    if not os.path.exists(d):
        !git clone --depth 1 https://github.com/{info["repo"]}.git {d}
    else:
        print(f"{d} already cloned")

In [ ]:
def load_gold_treebank(directory, filename):
    """Load a gold CoNLL-U file and return parsed sentences."""
    path = os.path.join(directory, filename)
    with open(path, "r", encoding="utf-8") as f:
        data = f.read()
    sentences = conllu.parse(data)
    return sentences


def extract_texts(sentences):
    """Extract the # text = metadata from each sentence."""
    texts = []
    for sent in sentences:
        text = sent.metadata.get("text", None)
        if text:
            texts.append(text)
        else:
            # Reconstruct from tokens as fallback
            tokens = [t["form"] for t in sent if isinstance(t["id"], int)]
            texts.append(" ".join(tokens))
    return texts


gold_data = {}
gold_texts = {}
for name, info in TREEBANKS.items():
    sents = load_gold_treebank(info["dir"], info["test_file"])
    gold_data[name] = sents
    gold_texts[name] = extract_texts(sents)
    print(f"{name}: {len(sents)} sentences loaded")

## 3. Re-parse with Each Parser

In [ ]:
# --- Stanza parsing ---

def parse_with_stanza(texts, lang, package):
    """Parse texts with Stanza full pipeline. Returns list of conllu-parsed sentences."""
    nlp = stanza.Pipeline(lang=lang, package=package, processors="tokenize,mwt,pos,lemma,depparse")
    results = []
    for i, text in enumerate(texts):
        if i % 100 == 0 and i > 0:
            print(f"  Stanza: {i}/{len(texts)}")
        doc = nlp(text)
        conllu_str = doc.to_conll()
        parsed = conllu.parse(conllu_str)
        results.append(parsed)
    return results


stanza_results = {}
for name, info in TREEBANKS.items():
    print(f"Parsing {name} with Stanza ({info['stanza_lang']}/{info['stanza_package']})...")
    stanza_results[name] = parse_with_stanza(
        gold_texts[name], info["stanza_lang"], info["stanza_package"]
    )
    print(f"  Done: {len(stanza_results[name])} sentences")

In [ ]:
# --- UDPipe 2 parsing via REST API ---

import time


def parse_with_udpipe(texts, model_name, batch_size=50):
    """Parse texts with UDPipe 2 REST API. Batches texts to reduce API calls."""
    results = []
    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start:batch_start + batch_size]
        # UDPipe API accepts one text at a time; batch by joining with double newline
        # and relying on the tokeniser. But for alignment we need per-sentence results,
        # so we send one at a time (with rate limiting).
        for i, text in enumerate(batch):
            idx = batch_start + i
            if idx % 100 == 0 and idx > 0:
                print(f"  UDPipe: {idx}/{len(texts)}")
            try:
                resp = requests.post(
                    f"{UDPIPE_API}/process",
                    data={
                        "model": model_name,
                        "tokenizer": "",
                        "tagger": "",
                        "parser": "",
                        "data": text,
                    },
                    timeout=30,
                )
                resp.raise_for_status()
                result_json = resp.json()
                conllu_str = result_json.get("result", "")
                parsed = conllu.parse(conllu_str)
                results.append(parsed)
            except Exception as e:
                print(f"  Error at sentence {idx}: {e}")
                results.append([])
            # Be polite to the API
            time.sleep(0.1)
    return results


udpipe_results = {}
for name, info in TREEBANKS.items():
    print(f"Parsing {name} with UDPipe 2 ({info['udpipe_model']})...")
    udpipe_results[name] = parse_with_udpipe(gold_texts[name], info["udpipe_model"])
    print(f"  Done: {len(udpipe_results[name])} sentences")

## 4. Alignment

Since parsers run full pipeline from raw text, tokenisation may differ from gold.
We align parser tokens to gold tokens using character offsets.

In [ ]:
def get_token_spans(tokens, text):
    """
    Compute character spans for a list of token forms against the original text.
    Returns list of (start, end, token_dict) tuples.
    """
    spans = []
    pos = 0
    for tok in tokens:
        form = tok["form"]
        idx = text.find(form, pos)
        if idx == -1:
            # Fuzzy fallback: try case-insensitive
            idx_lower = text.lower().find(form.lower(), pos)
            if idx_lower != -1:
                idx = idx_lower
            else:
                # Can't find token; use current position
                idx = pos
        spans.append((idx, idx + len(form), tok))
        pos = idx + len(form)
    return spans


def align_sentences(gold_sent, pred_sent, text):
    """
    Align gold and predicted tokens by character offset.
    Returns list of (gold_tok, pred_tok) pairs.
    Tokens that don't align get None for the missing side.
    """
    # Filter to word tokens only (skip MWTs with range IDs)
    gold_tokens = [t for t in gold_sent if isinstance(t["id"], int)]
    pred_tokens = [t for t in pred_sent if isinstance(t["id"], int)]

    gold_spans = get_token_spans(gold_tokens, text)
    pred_spans = get_token_spans(pred_tokens, text)

    aligned = []
    gi, pi = 0, 0

    while gi < len(gold_spans) and pi < len(pred_spans):
        g_start, g_end, g_tok = gold_spans[gi]
        p_start, p_end, p_tok = pred_spans[pi]

        if g_start == p_start and g_end == p_end:
            # Exact match
            aligned.append((g_tok, p_tok))
            gi += 1
            pi += 1
        elif g_end <= p_start:
            # Gold token has no pred match
            aligned.append((g_tok, None))
            gi += 1
        elif p_end <= g_start:
            # Pred token has no gold match
            aligned.append((None, p_tok))
            pi += 1
        else:
            # Overlapping but not identical — tokenisation mismatch
            aligned.append((g_tok, p_tok))
            gi += 1
            pi += 1

    # Remaining tokens
    while gi < len(gold_spans):
        aligned.append((gold_spans[gi][2], None))
        gi += 1
    while pi < len(pred_spans):
        aligned.append((None, pred_spans[pi][2]))
        pi += 1

    return aligned


def align_treebank(gold_sents, pred_results, texts):
    """
    Align an entire treebank. pred_results is a list where each element
    is a list of conllu-parsed sentences (a parser may split one input into multiple).
    """
    all_aligned = []
    tok_match_count = 0
    tok_mismatch_count = 0

    for i, (gold_sent, pred_sents, text) in enumerate(zip(gold_sents, pred_results, texts)):
        # Flatten predicted sentences (parser might split differently)
        if not pred_sents:
            all_aligned.append([])
            continue
        pred_tokens = []
        for ps in pred_sents:
            pred_tokens.extend(ps)

        aligned = align_sentences(gold_sent, pred_tokens, text)
        all_aligned.append(aligned)

        for g, p in aligned:
            if g is not None and p is not None and g["form"] == p["form"]:
                tok_match_count += 1
            else:
                tok_mismatch_count += 1

    total = tok_match_count + tok_mismatch_count
    if total > 0:
        print(f"  Token alignment: {tok_match_count}/{total} exact matches "
              f"({100*tok_match_count/total:.1f}%), {tok_mismatch_count} mismatches")
    return all_aligned

In [ ]:
stanza_aligned = {}
udpipe_aligned = {}

for name in TREEBANKS:
    print(f"\nAligning {name} — Stanza:")
    stanza_aligned[name] = align_treebank(
        gold_data[name], stanza_results[name], gold_texts[name]
    )
    print(f"Aligning {name} — UDPipe:")
    udpipe_aligned[name] = align_treebank(
        gold_data[name], udpipe_results[name], gold_texts[name]
    )

## 5. Evaluation — Parser vs Gold

In [ ]:
def evaluate_aligned(aligned_sents):
    """
    Evaluate aligned parser output against gold.
    Returns a dict of metrics and a detailed error DataFrame.
    """
    upos_correct = 0
    upos_total = 0
    lemma_correct = 0
    lemma_total = 0
    feats_correct = 0
    feats_total = 0
    tok_match = 0
    tok_mismatch = 0

    upos_confusion = Counter()
    lemma_errors = []
    pos_errors = []
    feat_errors = []

    for sent_idx, aligned in enumerate(aligned_sents):
        for g_tok, p_tok in aligned:
            if g_tok is None or p_tok is None:
                tok_mismatch += 1
                continue

            if g_tok["form"] != p_tok["form"]:
                tok_mismatch += 1
                continue

            tok_match += 1

            # UPOS
            g_upos = g_tok.get("upos", "_")
            p_upos = p_tok.get("upos", "_")
            upos_total += 1
            if g_upos == p_upos:
                upos_correct += 1
            else:
                upos_confusion[(g_upos, p_upos)] += 1
                pos_errors.append({
                    "sent_idx": sent_idx,
                    "form": g_tok["form"],
                    "gold_upos": g_upos,
                    "pred_upos": p_upos,
                })

            # Lemma
            g_lemma = g_tok.get("lemma", "_")
            p_lemma = p_tok.get("lemma", "_")
            lemma_total += 1
            if g_lemma == p_lemma:
                lemma_correct += 1
            else:
                lemma_errors.append({
                    "sent_idx": sent_idx,
                    "form": g_tok["form"],
                    "gold_lemma": g_lemma,
                    "pred_lemma": p_lemma,
                    "upos": g_upos,
                })

            # Morphological features
            g_feats = g_tok.get("feats") or {}
            p_feats = p_tok.get("feats") or {}
            feats_total += 1
            if g_feats == p_feats:
                feats_correct += 1
            else:
                all_keys = set(g_feats.keys()) | set(p_feats.keys())
                for k in all_keys:
                    gv = g_feats.get(k)
                    pv = p_feats.get(k)
                    if gv != pv:
                        feat_errors.append({
                            "sent_idx": sent_idx,
                            "form": g_tok["form"],
                            "feature": k,
                            "gold_value": str(gv) if gv else "MISSING",
                            "pred_value": str(pv) if pv else "MISSING",
                            "upos": g_upos,
                        })

    metrics = {
        "tok_match": tok_match,
        "tok_mismatch": tok_mismatch,
        "upos_accuracy": upos_correct / upos_total if upos_total else 0,
        "upos_correct": upos_correct,
        "upos_total": upos_total,
        "lemma_accuracy": lemma_correct / lemma_total if lemma_total else 0,
        "lemma_correct": lemma_correct,
        "lemma_total": lemma_total,
        "feats_accuracy": feats_correct / feats_total if feats_total else 0,
        "feats_correct": feats_correct,
        "feats_total": feats_total,
    }

    return metrics, upos_confusion, pd.DataFrame(pos_errors), pd.DataFrame(lemma_errors), pd.DataFrame(feat_errors)

In [ ]:
# Evaluate all treebanks for both parsers
eval_results = {}

for name in TREEBANKS:
    eval_results[name] = {}

    print(f"\n{'='*60}")
    print(f"Treebank: {name}")
    print(f"{'='*60}")

    for parser_name, aligned_data in [("stanza", stanza_aligned), ("udpipe", udpipe_aligned)]:
        metrics, confusion, pos_df, lemma_df, feat_df = evaluate_aligned(aligned_data[name])
        eval_results[name][parser_name] = {
            "metrics": metrics,
            "confusion": confusion,
            "pos_errors": pos_df,
            "lemma_errors": lemma_df,
            "feat_errors": feat_df,
        }

        print(f"\n  {parser_name.upper()}:")
        print(f"    Tokens aligned: {metrics['tok_match']}, mismatched: {metrics['tok_mismatch']}")
        print(f"    UPOS accuracy:  {metrics['upos_accuracy']:.4f} ({metrics['upos_correct']}/{metrics['upos_total']})")
        print(f"    Lemma accuracy: {metrics['lemma_accuracy']:.4f} ({metrics['lemma_correct']}/{metrics['lemma_total']})")
        print(f"    Feats accuracy: {metrics['feats_accuracy']:.4f} ({metrics['feats_correct']}/{metrics['feats_total']})")

        if confusion:
            print(f"    Top UPOS confusions:")
            for (g, p), count in confusion.most_common(10):
                print(f"      {g} -> {p}: {count}")

In [ ]:
# Summary table
summary_rows = []
for name in TREEBANKS:
    for parser in ["stanza", "udpipe"]:
        m = eval_results[name][parser]["metrics"]
        summary_rows.append({
            "treebank": name,
            "parser": parser,
            "tok_match": m["tok_match"],
            "tok_mismatch": m["tok_mismatch"],
            "upos_acc": f"{m['upos_accuracy']:.4f}",
            "lemma_acc": f"{m['lemma_accuracy']:.4f}",
            "feats_acc": f"{m['feats_accuracy']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# Detailed POS error patterns per treebank
for name in TREEBANKS:
    for parser in ["stanza", "udpipe"]:
        pos_df = eval_results[name][parser]["pos_errors"]
        if len(pos_df) > 0:
            print(f"\n{name} / {parser} — Top POS error patterns:")
            pattern_counts = pos_df.groupby(["gold_upos", "pred_upos"]).size().reset_index(name="count")
            pattern_counts = pattern_counts.sort_values("count", ascending=False).head(15)
            display(pattern_counts)

In [ ]:
# Lemma error analysis by POS
for name in TREEBANKS:
    for parser in ["stanza", "udpipe"]:
        lemma_df = eval_results[name][parser]["lemma_errors"]
        if len(lemma_df) > 0:
            print(f"\n{name} / {parser} — Lemma errors by POS:")
            by_pos = lemma_df.groupby("upos").size().reset_index(name="count")
            by_pos = by_pos.sort_values("count", ascending=False)
            display(by_pos.head(10))
            print(f"  Sample lemma errors:")
            display(lemma_df.head(10))

## 6. Cross-parser Comparison

Categorise each token into:
- **Both correct**: both parsers agree with gold
- **Both wrong, same error**: both parsers agree, differ from gold — potential gold issue
- **Both wrong, different**: genuinely hard case
- **Stanza correct only**: correctable UDPipe error
- **UDPipe correct only**: correctable Stanza error

In [ ]:
def cross_compare(stanza_aligned_sents, udpipe_aligned_sents, gold_sents, texts, field="upos"):
    """
    Cross-compare two parser outputs for a given field.
    Only considers tokens where both parsers have an aligned token
    with matching form to the gold.
    """
    categories = {
        "both_correct": 0,
        "both_wrong_same": 0,
        "both_wrong_diff": 0,
        "stanza_only_correct": 0,
        "udpipe_only_correct": 0,
    }
    details = []

    for sent_idx in range(min(len(stanza_aligned_sents), len(udpipe_aligned_sents))):
        s_aligned = stanza_aligned_sents[sent_idx]
        u_aligned = udpipe_aligned_sents[sent_idx]

        # Build lookup from gold token form+position to parser predictions
        s_by_form = {}
        for g, p in s_aligned:
            if g is not None and p is not None and g["form"] == p["form"]:
                key = (sent_idx, g["id"], g["form"])
                s_by_form[key] = (g, p)

        u_by_form = {}
        for g, p in u_aligned:
            if g is not None and p is not None and g["form"] == p["form"]:
                key = (sent_idx, g["id"], g["form"])
                u_by_form[key] = (g, p)

        common_keys = set(s_by_form.keys()) & set(u_by_form.keys())

        for key in common_keys:
            g_tok_s, s_tok = s_by_form[key]
            g_tok_u, u_tok = u_by_form[key]

            def get_field_val(tok):
                if field == "feats":
                    return str(tok.get("feats") or {})
                return tok.get(field, "_")

            gold_val = get_field_val(g_tok_s)
            stanza_val = get_field_val(s_tok)
            udpipe_val = get_field_val(u_tok)

            s_correct = (stanza_val == gold_val)
            u_correct = (udpipe_val == gold_val)

            if s_correct and u_correct:
                categories["both_correct"] += 1
            elif s_correct and not u_correct:
                categories["stanza_only_correct"] += 1
                details.append({
                    "sent_idx": sent_idx,
                    "form": g_tok_s["form"],
                    "field": field,
                    "gold": gold_val,
                    "stanza": stanza_val,
                    "udpipe": udpipe_val,
                    "category": "stanza_only_correct",
                })
            elif not s_correct and u_correct:
                categories["udpipe_only_correct"] += 1
                details.append({
                    "sent_idx": sent_idx,
                    "form": g_tok_s["form"],
                    "field": field,
                    "gold": gold_val,
                    "stanza": stanza_val,
                    "udpipe": udpipe_val,
                    "category": "udpipe_only_correct",
                })
            elif stanza_val == udpipe_val:
                categories["both_wrong_same"] += 1
                details.append({
                    "sent_idx": sent_idx,
                    "form": g_tok_s["form"],
                    "field": field,
                    "gold": gold_val,
                    "stanza": stanza_val,
                    "udpipe": udpipe_val,
                    "category": "both_wrong_same",
                })
            else:
                categories["both_wrong_diff"] += 1
                details.append({
                    "sent_idx": sent_idx,
                    "form": g_tok_s["form"],
                    "field": field,
                    "gold": gold_val,
                    "stanza": stanza_val,
                    "udpipe": udpipe_val,
                    "category": "both_wrong_diff",
                })

    return categories, pd.DataFrame(details)

In [ ]:
cross_results = {}

for name in TREEBANKS:
    cross_results[name] = {}
    print(f"\n{'='*60}")
    print(f"Cross-comparison: {name}")
    print(f"{'='*60}")

    for field in ["upos", "lemma"]:
        cats, details_df = cross_compare(
            stanza_aligned[name], udpipe_aligned[name],
            gold_data[name], gold_texts[name], field=field
        )
        cross_results[name][field] = {"categories": cats, "details": details_df}

        total = sum(cats.values())
        print(f"\n  {field.upper()}:")
        for cat, count in cats.items():
            pct = 100 * count / total if total else 0
            print(f"    {cat}: {count} ({pct:.1f}%)")

        # Show potential gold annotation issues
        if len(details_df) > 0:
            both_wrong_same = details_df[details_df["category"] == "both_wrong_same"]
            if len(both_wrong_same) > 0:
                print(f"\n    Potential gold issues (both parsers agree, differ from gold):")
                pattern_counts = both_wrong_same.groupby(["gold", "stanza"]).size().reset_index(name="count")
                pattern_counts = pattern_counts.sort_values("count", ascending=False)
                display(pattern_counts.head(10))

In [ ]:
# Identify systematic correction patterns
print("SYSTEMATIC CORRECTION PATTERNS")
print("="*60)

for name in TREEBANKS:
    print(f"\nTreebank: {name}")
    for field in ["upos", "lemma"]:
        details_df = cross_results[name][field]["details"]
        if len(details_df) == 0:
            continue

        # Cases where one parser is systematically better
        for parser, cat in [("stanza", "stanza_only_correct"), ("udpipe", "udpipe_only_correct")]:
            subset = details_df[details_df["category"] == cat]
            if len(subset) == 0:
                continue

            other = "udpipe" if parser == "stanza" else "stanza"
            print(f"\n  {field.upper()} — {parser} correct, {other} wrong ({len(subset)} cases):")

            # Group by error pattern
            patterns = subset.groupby(["gold", other]).size().reset_index(name="count")
            patterns = patterns.sort_values("count", ascending=False)
            display(patterns.head(10))

## 7. Visualisation & Export

In [ ]:
# UPOS confusion matrix heatmaps
ALL_UPOS = [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN",
    "NUM", "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X"
]


def plot_confusion_matrix(confusion_counter, title, ax):
    """Plot a UPOS confusion matrix from a Counter of (gold, pred) pairs."""
    # Get tags that actually appear
    tags = sorted(set(
        [g for g, _ in confusion_counter.keys()] +
        [p for _, p in confusion_counter.keys()]
    ))
    if not tags:
        ax.set_title(f"{title}\n(no errors)")
        return

    matrix = pd.DataFrame(0, index=tags, columns=tags)
    for (g, p), count in confusion_counter.items():
        if g in matrix.index and p in matrix.columns:
            matrix.loc[g, p] = count

    sns.heatmap(matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax, cbar_kws={"shrink": 0.8})
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Gold")


for name in TREEBANKS:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f"UPOS Confusion — {name}", fontsize=14)

    for i, parser in enumerate(["stanza", "udpipe"]):
        confusion = eval_results[name][parser]["confusion"]
        plot_confusion_matrix(confusion, f"{parser.upper()}", axes[i])

    plt.tight_layout()
    plt.savefig(f"confusion_{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Agreement/disagreement summary bar chart
fig, axes = plt.subplots(1, len(TREEBANKS), figsize=(6 * len(TREEBANKS), 5))
if len(TREEBANKS) == 1:
    axes = [axes]

category_labels = {
    "both_correct": "Both correct",
    "stanza_only_correct": "Stanza only",
    "udpipe_only_correct": "UDPipe only",
    "both_wrong_same": "Both wrong\n(agree)",
    "both_wrong_diff": "Both wrong\n(disagree)",
}
cat_colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c", "#9b59b6"]

for idx, name in enumerate(TREEBANKS):
    cats = cross_results[name]["upos"]["categories"]
    labels = [category_labels[k] for k in cats.keys()]
    values = list(cats.values())

    axes[idx].bar(labels, values, color=cat_colors)
    axes[idx].set_title(f"{name} — UPOS")
    axes[idx].set_ylabel("Token count")
    for j, v in enumerate(values):
        axes[idx].text(j, v + max(values) * 0.01, str(v), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("cross_comparison_upos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Export error tables as CSV
os.makedirs("output", exist_ok=True)

for name in TREEBANKS:
    for parser in ["stanza", "udpipe"]:
        for error_type in ["pos_errors", "lemma_errors", "feat_errors"]:
            df = eval_results[name][parser][error_type]
            if len(df) > 0:
                fname = f"output/{name}_{parser}_{error_type}.csv"
                df.to_csv(fname, index=False)
                print(f"Saved {fname} ({len(df)} rows)")

    for field in ["upos", "lemma"]:
        details_df = cross_results[name][field]["details"]
        if len(details_df) > 0:
            fname = f"output/{name}_cross_{field}.csv"
            details_df.to_csv(fname, index=False)
            print(f"Saved {fname} ({len(details_df)} rows)")

print("\nDone! All error tables exported.")

In [ ]:
# Spot-check: show a few sentences with their gold and parser annotations
for name in list(TREEBANKS.keys())[:1]:  # Just first treebank
    print(f"\nSpot-check: {name}")
    print("=" * 60)
    for i in range(min(3, len(gold_data[name]))):
        text = gold_texts[name][i]
        print(f"\nSentence {i}: {text}")
        print(f"{'Token':<20} {'Gold POS':<10} {'Stanza POS':<12} {'UDPipe POS':<12} {'Gold Lemma':<15} {'Stanza Lemma':<15} {'UDPipe Lemma':<15}")
        print("-" * 100)

        # Build lookup from aligned data
        s_aligned = stanza_aligned[name][i]
        u_aligned = udpipe_aligned[name][i]

        gold_tokens = [t for t in gold_data[name][i] if isinstance(t["id"], int)]

        s_lookup = {}
        for g, p in s_aligned:
            if g is not None and p is not None:
                s_lookup[g["form"], g["id"]] = p

        u_lookup = {}
        for g, p in u_aligned:
            if g is not None and p is not None:
                u_lookup[g["form"], g["id"]] = p

        for tok in gold_tokens:
            form = tok["form"]
            g_upos = tok.get("upos", "_")
            g_lemma = tok.get("lemma", "_")

            s_tok = s_lookup.get((form, tok["id"]))
            u_tok = u_lookup.get((form, tok["id"]))

            s_upos = s_tok.get("upos", "_") if s_tok else "—"
            u_upos = u_tok.get("upos", "_") if u_tok else "—"
            s_lemma = s_tok.get("lemma", "_") if s_tok else "—"
            u_lemma = u_tok.get("lemma", "_") if u_tok else "—"

            # Mark mismatches
            s_upos_mark = f"*{s_upos}*" if s_upos != g_upos and s_upos != "—" else s_upos
            u_upos_mark = f"*{u_upos}*" if u_upos != g_upos and u_upos != "—" else u_upos
            s_lemma_mark = f"*{s_lemma}*" if s_lemma != g_lemma and s_lemma != "—" else s_lemma
            u_lemma_mark = f"*{u_lemma}*" if u_lemma != g_lemma and u_lemma != "—" else u_lemma

            print(f"{form:<20} {g_upos:<10} {s_upos_mark:<12} {u_upos_mark:<12} {g_lemma:<15} {s_lemma_mark:<15} {u_lemma_mark:<15}")